# 1. Khai báo đường dẫn và cấu hình


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

TARGET = "Attack_type"
CATEGORICAL_FEATURES = ["proto", "service"]
EXCLUDED_COLUMNS = [TARGET, "Unnamed: 0", "group_id"]

CURRENT = Path.cwd().resolve()
ROOT = next(
    (
        parent
        for parent in [CURRENT, *CURRENT.parents]
        if (parent / "project").is_dir()
        and (parent / "splitted data").is_dir()
    ),
    None,
)
if ROOT is None:
    raise FileNotFoundError("Không tìm thấy thư mục gốc sample_project")

SOURCE_DIR = ROOT / "splitted data"
TRAIN_SOURCE = SOURCE_DIR / "train.csv"
VALIDATION_SOURCE = SOURCE_DIR / "validation.csv"

print("Thực nghiệm:", "E13")
print("Nguồn Train:", TRAIN_SOURCE)
print("Nguồn Validation:", VALIDATION_SOURCE)


# 2. Đọc Train và Validation đã chia sẵn


In [ ]:
for data_path in (TRAIN_SOURCE, VALIDATION_SOURCE):
    if not data_path.is_file():
        raise FileNotFoundError(data_path)

train_df = pd.read_csv(TRAIN_SOURCE)
validation_df = pd.read_csv(VALIDATION_SOURCE)

if TARGET not in train_df.columns or TARGET not in validation_df.columns:
    raise ValueError(f"Train và Validation phải có cột nhãn {TARGET!r}")
if not train_df.columns.equals(validation_df.columns):
    raise ValueError("Tên hoặc thứ tự cột Train/Validation không khớp")
if train_df.empty or validation_df.empty:
    raise ValueError("Train và Validation không được để trống")
if train_df[TARGET].isna().any() or validation_df[TARGET].isna().any():
    raise ValueError("Nhãn có giá trị thiếu")

print("Train:", train_df.shape)
print("Validation:", validation_df.shape)
display(train_df.head(3))


# 3. Phân loại đặc trưng và kiểm tra dữ liệu


In [ ]:
feature_columns = [
    column for column in train_df.columns if column not in EXCLUDED_COLUMNS
]
missing_categories = [
    column for column in CATEGORICAL_FEATURES if column not in feature_columns
]
if missing_categories:
    raise ValueError(f"Thiếu đặc trưng phân loại: {missing_categories}")

numeric_features = (
    train_df[feature_columns].select_dtypes(include=np.number).columns.tolist()
)
unknown_features = sorted(
    set(feature_columns) - set(numeric_features) - set(CATEGORICAL_FEATURES)
)
if unknown_features:
    raise ValueError(f"Có đặc trưng chưa được phân loại: {unknown_features}")

for frame, label in ((train_df, "Train"), (validation_df, "Validation")):
    if frame[feature_columns].isna().any().any():
        raise ValueError(f"{label} có giá trị thiếu")
    if not np.isfinite(frame[numeric_features].to_numpy(dtype=float)).all():
        raise ValueError(f"{label} có giá trị số NaN/Inf")

labels = sorted(train_df[TARGET].astype(str).unique().tolist())
unknown_labels = sorted(
    set(validation_df[TARGET].astype(str).unique()) - set(labels)
)
if unknown_labels:
    raise ValueError(f"Validation có nhãn không xuất hiện trong Train: {unknown_labels}")

feature_summary = pd.DataFrame(
    [
        {"Nhóm đặc trưng": "Numeric", "Số lượng": len(numeric_features)},
        {
            "Nhóm đặc trưng": "Categorical",
            "Số lượng": len(CATEGORICAL_FEATURES),
        },
        {"Nhóm đặc trưng": "Nhãn", "Số lượng": 1},
    ]
)
print("Số nhãn:", len(labels))
display(feature_summary)
display(train_df[TARGET].value_counts().rename("Số mẫu Train").to_frame())


# 4. Tiền xử lý riêng cho E13

Áp dụng Signed-log cho các đặc trưng số.

Không biến đổi `Attack_type`. `proto` và `service` được giữ nguyên; OneHotEncoder được fit trong Pipeline của thực nghiệm ở từng fold CV.


In [ ]:
def signed_log(values):
    values = np.asarray(values, dtype=np.float64)
    return np.sign(values) * np.log1p(np.abs(values))


train_processed = train_df.copy()
validation_processed = validation_df.copy()
train_before = train_df[numeric_features].head(5).copy()
validation_before = validation_df[numeric_features].head(5).copy()

train_processed.loc[:, numeric_features] = signed_log(
    train_processed[numeric_features].to_numpy(dtype=np.float64)
)
validation_processed.loc[:, numeric_features] = signed_log(
    validation_processed[numeric_features].to_numpy(dtype=np.float64)
)

preview = pd.DataFrame(
    {
        "Trước Signed-log": train_before.iloc[:, 0],
        "Sau Signed-log": train_processed[numeric_features].head(5).iloc[:, 0],
    }
)
print("Đặc trưng số được xử lý:", len(numeric_features))
print("Cột phân loại được giữ nguyên:", CATEGORICAL_FEATURES)
display(preview)


# 5. Kiểm tra dữ liệu sau tiền xử lý


In [ ]:
if not train_processed.columns.equals(train_df.columns):
    raise ValueError("Tiền xử lý đã làm thay đổi cấu trúc cột Train")
if not validation_processed.columns.equals(validation_df.columns):
    raise ValueError("Tiền xử lý đã làm thay đổi cấu trúc cột Validation")
if not train_processed[TARGET].equals(train_df[TARGET]):
    raise ValueError("Không được biến đổi cột nhãn Train")
if not validation_processed[TARGET].equals(validation_df[TARGET]):
    raise ValueError("Không được biến đổi cột nhãn Validation")

for column in CATEGORICAL_FEATURES:
    if not train_processed[column].equals(train_df[column]):
        raise ValueError(f"Không được biến đổi cột phân loại Train: {column}")
    if not validation_processed[column].equals(validation_df[column]):
        raise ValueError(f"Không được biến đổi cột phân loại Validation: {column}")

if not np.isfinite(train_processed[numeric_features].to_numpy(dtype=float)).all():
    raise ValueError("Train sau xử lý có giá trị số NaN/Inf")
if not np.isfinite(
    validation_processed[numeric_features].to_numpy(dtype=float)
).all():
    raise ValueError("Validation sau xử lý có giá trị số NaN/Inf")

if "signed_log" == "none":
    if not np.array_equal(
        train_processed[numeric_features].to_numpy(),
        train_df[numeric_features].to_numpy(),
    ):
        raise ValueError("E11 phải giữ nguyên đặc trưng số")
else:
    expected = signed_log(train_df[numeric_features].to_numpy(dtype=np.float64))
    if not np.allclose(
        train_processed[numeric_features].to_numpy(dtype=np.float64),
        expected,
    ):
        raise ValueError("Kết quả Signed-log không khớp công thức")

check_summary = pd.DataFrame(
    [
        {"Kiểm tra": "Số dòng Train", "Kết quả": len(train_processed)},
        {"Kiểm tra": "Số dòng Validation", "Kết quả": len(validation_processed)},
        {"Kiểm tra": "Số đặc trưng số", "Kết quả": len(numeric_features)},
        {"Kiểm tra": "Số đặc trưng phân loại", "Kết quả": len(CATEGORICAL_FEATURES)},
        {"Kiểm tra": "Số nhãn", "Kết quả": len(labels)},
        {"Kiểm tra": "Giá trị thiếu/NaN/Inf", "Kết quả": "Không phát hiện"},
    ]
)
display(check_summary)


# 6. Lưu Train/Validation đã xử lý

Notebook thực nghiệm E tương ứng đọc trực tiếp Train/Validation từ `splitted data` và thực hiện cùng phương pháp trong bộ nhớ. Scaler (nếu có) vẫn được fit trong từng fold CV để tránh rò rỉ dữ liệu.


In [ ]:
display(
    pd.DataFrame(
        [
            {"Tập": "Train", "Số dòng": len(train_processed), "Số cột": len(train_processed.columns)},
            {"Tập": "Validation", "Số dòng": len(validation_processed), "Số cột": len(validation_processed.columns)},
        ]
    )
)
print("Hoàn tất tiền xử lý trong bộ nhớ; notebook không ghi file vào result.")
